# Mushroom Dataset - XGBoost

## Contributors
- Liam: XGBoost training, tuning, validation, and metrics
- Pam: Review

This notebook trains and tunes an XGBoost classifier independently from the baseline and Random Forest notebooks.

## Evaluation design

An 80/20 stratified split creates the training and holdout sets. Randomized hyperparameter tuning uses 5-fold cross-validation only on the training set; the holdout test set is used once for final validation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
df = pd.read_csv("mushroom_cleaned.csv")
target = "class"
X = df.drop(columns=target)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train shape: {X_train.shape}; test shape: {X_test.shape}")

In [ ]:
from pathlib import Path
from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

numeric = X_train.select_dtypes(include="number").columns.tolist()
categorical = X_train.select_dtypes(exclude="number").columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median", add_indicator=True), numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical),
])
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(objective="binary:logistic", eval_metric="logloss", tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1)),
])
search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions={
        "model__n_estimators": [150, 250, 400],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.03, 0.08, 0.15],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0],
    },
    n_iter=15,
    scoring="balanced_accuracy",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
)
search.fit(X_train, (y_train == "p").astype(int))
xgb_final = search.best_estimator_
xgb_labels = np.where(xgb_final.predict(X_test) == 1, "p", "e")
xgb_probabilities = xgb_final.predict_proba(X_test)[:, 1]
xgb_report = classification_report(y_test, xgb_labels, target_names=["edible", "poisonous"], output_dict=True)
xgb_metrics = {
    "model": "Tuned XGBoost",
    "accuracy": accuracy_score(y_test, xgb_labels),
    "balanced_accuracy": balanced_accuracy_score(y_test, xgb_labels),
    "poisonous_precision": precision_score(y_test, xgb_labels, pos_label="p"),
    "poisonous_recall": recall_score(y_test, xgb_labels, pos_label="p"),
    "poisonous_f1": f1_score(y_test, xgb_labels, pos_label="p"),
    "roc_auc": roc_auc_score((y_test == "p").astype(int), xgb_probabilities),
}
print("Best parameters:", search.best_params_)
print(pd.Series(xgb_metrics))
print("Confusion matrix [edible, poisonous]:\\n", confusion_matrix(y_test, xgb_labels))
pd.DataFrame([xgb_metrics]).to_csv("xgboost_metrics.csv", index=False)
Path("xgboost_metrics.json").write_text(pd.Series({**xgb_metrics, "best_cv_balanced_accuracy": search.best_score_, "best_parameters": search.best_params_, "classification_report": xgb_report}).to_json(indent=2), encoding="utf-8")
dump(xgb_final, "xgboost_final.joblib")
print("Saved XGBoost model and metrics")

## Conclusion

XGBoost is evaluated using the same split and target definition as Random Forest. The final choice should emphasize balanced accuracy and poisonous recall, not accuracy alone.